## Submissions for speakers dataset

In [1]:
from preprocessing_class import Preprocessing, load_pres_test, load_pres
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression,LogisticRegressionCV
from sklearn.svm import LinearSVC,SVC
from nltk.corpus import stopwords
import numpy as np
import string
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2

#### Format submission
```
submission-pres-X.csv
```

In [2]:
# load dataset train
alltxt, alllabs = load_pres("Dataset/corpus.tache1.learn.utf8")
alltxt, alllabs = np.array(alltxt), np.array(alllabs)

In [3]:
# load dataset test
alltxt_test = load_pres_test("Test_set/corpus.tache1.test.utf8")
alltxt_test = np.array(alltxt_test)

In [4]:
alltxt_test.shape, alltxt.shape

((27162,), (57413,))

### Submission 1 (number 2)

In [5]:
def build_vectorizer(name="count", stop_words=None, max_features=None,
                     lowercase=False, strip_accents=None, ngram_range=(1,1), use_idf= True, smooth_idf=True, sublinear_tf=False, 
                     max_df=0.99, min_df=0.01, preprocessor = lambda x: x):
    if name == "count":
        vectorizer = CountVectorizer(stop_words=stop_words,max_features=max_features,max_df=max_df,min_df=min_df,ngram_range=ngram_range,lowercase=lowercase,
                                     strip_accents=strip_accents, preprocessor=preprocessor)    
        
    elif name == "tfidf":
        vectorizer = TfidfVectorizer(use_idf= use_idf, smooth_idf=smooth_idf, sublinear_tf=sublinear_tf,max_df=max_df, min_df=min_df, max_features=max_features,
                                     stop_words=stop_words, ngram_range=ngram_range,lowercase=lowercase,strip_accents=strip_accents, preprocessor=preprocessor)
    
    return vectorizer
    ## to implement more embeddings 

def build_embeddings():
    pass

def vectorize_txts(txts,vectorizer):
    X = vectorizer.fit_transform(txts)
    return X

def build_model(name="kmeans", n_clusters = 2, max_iter = 1000, penalty='l2', loss='squared_hinge',kernel='rbf',solver='lbfgs'):
    if name == "kmeans":
        return KMeans(n_clusters=n_clusters,max_iter=max_iter) 
    elif name == "linear_svm":
        return LinearSVC(penalty=penalty,loss=loss,max_iter=max_iter)
    elif name == "svm":
        return SVC(max_iter=max_iter,kernel=kernel)
    elif name == "logreg":
        return LogisticRegression(solver=solver,max_iter=max_iter) 

In [6]:
punc = set(string.punctuation + '\n\r\t')
punc.remove("'") # keep the ' for french words 
custom_punctuation = "".join(punc)

prep = Preprocessing(low_case = True,
                rm_punctuation = True,
                rm_number = False,
                word_norm = None, # stem faster
                pos_tagging = False, # to check
                all_capital = True, # keep all capital words as they are
                cap_name = True, # garder les noms en majuscules, for pos tag
                rm_accent = False, 
                lang = "french",
                punct = custom_punctuation, # punctuation can contain -, that would be kept
                urls = False 
                )

In [7]:
final_stopwords_list = stopwords.words('french') 
vectoriser = build_vectorizer(name="tfidf", stop_words=final_stopwords_list, max_features=None, preprocessor=prep)

# train on the sub train
vect_train = vectoriser.fit_transform(alltxt)
vect_test = vectoriser.transform(alltxt_test)

In [8]:
# model logistic regression
model = build_model(name="logreg")

In [9]:
model.fit(vect_train,alllabs)

# preds
y_pred_train = model.predict(vect_train)
ypred_train_prob = model.predict_proba(vect_train) # probas for ROC

y_pred = model.predict(vect_test)
y_pred_prob = model.predict_proba(vect_test)

In [13]:
from sklearn.metrics import average_precision_score, f1_score

average_precision_score(alllabs, y_pred_train, average="micro"), average_precision_score(alllabs, y_pred_train, average="weighted")

(0.8763620445868716, 0.8763620445868716)

In [15]:
f1_score(alllabs, y_pred_train, average="macro"), f1_score(alllabs, y_pred_train, average="weighted"),

(0.5289097284272443, 0.8263233694731392)

In [28]:
# dim 0 = -1, dim 1 = 1
# M = -1, C = 1
ypred_train_prob # keep dim 0  

array([[0.61477151, 0.38522849],
       [0.13165143, 0.86834857],
       [0.08209623, 0.91790377],
       ...,
       [0.02687995, 0.97312005],
       [0.31412181, 0.68587819],
       [0.07823031, 0.92176969]], shape=(57413, 2))

In [69]:
# function to use
from sklearn.metrics import f1_score

def save_submission(predictions, number):
    filename = f"Test_submission/pres/submission-pres-{number}.csv"
    np.savetxt(filename, predictions, delimiter="\n", fmt='%f')

def model_prediction(model, vectorizer, X_train, y_train, X_test, number = 'test'):
    # vectorizing
    vect_train = vectorizer.fit_transform(X_train)
    vect_test = vectorizer.transform(X_test)

    # scaling ?

    model.fit(vect_train, y_train)

    # predictions
    y_pred_train = model.predict(vect_train)
    y_pred_prob_train = model.predict_proba(vect_train)

    y_pred_prob = model.predict_proba(vect_test)
    predictions = y_pred_prob[:,0]

    # prints
    dist = np.where(predictions < 0.5, 'C', 'M')
    val, count = np.unique(dist, return_counts=True) # seems like unbalanced, or biaised

    print("F1 score of train",f1_score(y_train,y_pred_train))
    print("Distribution of test", val, count)

    # saving
    save_submission(predictions, number)

In [70]:
punc = set(string.punctuation + '\n\r\t')
punc.remove("'") # keep the ' for french words 
custom_punctuation = "".join(punc)

prep = Preprocessing(low_case = True,
                rm_punctuation = True,
                rm_number = False,
                word_norm = None, # stem faster
                pos_tagging = False, # to check
                all_capital = True, # keep all capital words as they are
                cap_name = True, # garder les noms en majuscules, for pos tag
                rm_accent = False, 
                lang = "french",
                punct = custom_punctuation, # punctuation can contain -, that would be kept
                urls = False 
                )

In [71]:
model = build_model(name="logreg")

final_stopwords_list = stopwords.words('french') 
vectoriser = build_vectorizer(name="tfidf", stop_words=final_stopwords_list, max_features=None, preprocessor=prep)

In [ ]:
model_prediction(model, vectoriser, alltxt, alllabs, alltxt_test, number=1)

F1 score of train 0.9319453309803627
Distribution of test ['C' 'M'] [26772   390]


In [3]:
(np.array([26772 ,390]))/np.sum(np.array([26772 ,390]))

array([0.98564171, 0.01435829])

### Submission 2 (number 6)